In [ ]:
# Cell 1: Setup - Import required libraries
from pyspark.sql import functions as F
from pyspark.sql.types import *
from datetime import datetime
import uuid

In [ ]:
# Cell 2: Define catalog variables
# source_catalog: bronze layer (not used directly - source is star schema)
source_catalog = "bronze_non_prod"
# target_catalog: enriched/temporary layer for temp tables
target_catalog = "djj_enriched_non_prod"
# federated_starschema_catalog: star schema layer (DJJStarSchema -> djj_starschema)
federated_starschema_catalog = "djj_starschema"

In [ ]:
# Cell 3: Define widget parameters for ETL configuration variables
dbutils.widgets.text("MasterSourceSystemName", "DJJStarSchema")
dbutils.widgets.text("SourceSubSystemName", "")
dbutils.widgets.text("SourceDatabaseName", "DJJStarSchema")
dbutils.widgets.text("SourceTableName", "Brk.factOpenOrderSnapshot")
dbutils.widgets.text("DestinationDatabaseName", "DJJStarSchema")
dbutils.widgets.text("DestinationTableName", "factOpenOrderSnapshot")
dbutils.widgets.text("ETLType", "Incremental")
dbutils.widgets.text("PkgName", "NUE.etl_factOpenOrderSnapshot_MDSCargoAdjustments")
dbutils.widgets.text("ETLTemplate", "")

In [ ]:
# Cell 4: Retrieve widget parameter values
MasterSourceSystemName = dbutils.widgets.get("MasterSourceSystemName")
SourceSubSystemName = dbutils.widgets.get("SourceSubSystemName")
SourceDatabaseName = dbutils.widgets.get("SourceDatabaseName")
SourceTableName = dbutils.widgets.get("SourceTableName")
DestinationDatabaseName = dbutils.widgets.get("DestinationDatabaseName")
DestinationTableName = dbutils.widgets.get("DestinationTableName")
ETLType = dbutils.widgets.get("ETLType")
PkgName = dbutils.widgets.get("PkgName")
ETLTemplate = dbutils.widgets.get("ETLTemplate")
EnrichedTimestampUTC = datetime.utcnow().strftime("%Y-%m-%d %H:%M:%S")

In [ ]:
# Cell 5: Initialize ETL tracking variables
PkgGUID = str(uuid.uuid4())
SSIDGUID = str(uuid.uuid4())
ETLRunTime = datetime.utcnow().strftime("%Y-%m-%d %H:%M:%S")
IncrementalExtractTime = datetime.utcnow().strftime("%Y-%m-%d %H:%M:%S")
RowsInserted = 0
RowsUpdated = 0
RowsDeleted = 0
ErrorRowCount = 0
TableInitialRowCount = 0
TableFinalRowCount = 0
ETLStatus = ''
ExtractRowCount = 0

In [ ]:
# Cell 6: Insert/Update records in ETL Master
logger = ETLLogger(spark)

ETLID = logger.log_etl(
    MasterSourceSystemName=MasterSourceSystemName,
    SourceSubSystemName=SourceSubSystemName,
    SourceDatabaseName=SourceDatabaseName,
    SourceTableName=SourceTableName,
    DestinationDatabaseName=DestinationDatabaseName,
    DestinationTableName=DestinationTableName,
    PkgName=PkgName,
    PkgGUID=PkgGUID,
    ETLTemplate=ETLTemplate,
    ETLType=ETLType,
    CheckpointsEnabled=True
)

print(ETLID)

In [ ]:
# Cell 7: Drop temp tables before processing
spark.sql(f"DROP TABLE IF EXISTS {target_catalog}.temp.NucorMDSCargoAdjustments")

In [ ]:
# Cell 8: Create temp table with source query
# Source: Brk.factOpenOrderSnapShot -> {federated_starschema_catalog}.djj_brk.factOpenOrderSnapShot
# DJJLastUpdateTime in starschema -> DJJLastUpdateTime (unchanged)
spark.sql(f"""
    CREATE TABLE {target_catalog}.temp.NucorMDSCargoAdjustments
    USING DELTA
    AS
    SELECT
        e.Fiscal_Week_of_Year AS FiscalWeekKey,
        a.DateKey,
        Company AS BizType,
        CASE WHEN lower(trim(a.ContractType)) = lower(trim('P')) THEN 'Purchase' ELSE 'Sale' END AS ContractType,
        ContractNum,
        ContractLine,
        SourceSystemKey,
        NucorOrgID,
        ShippingScenarioID,
        CAST(NULL AS INT) AS InputDateKey,
        CAST(NULL AS INT) AS OrderDateKey,
        CAST(NULL AS INT) AS BuyPlanMonthKey,
        a.GradeKey,
        0 AS SupplierID,
        ConsumerID,
        GradeCode,
        OrderPropertiesKey,
        a.InventoryLocationKey,
        VehicleTypeKey,
        car.CargoID,
        car.StatusDesc AS StatusDesc,
        'Not Shared' AS SharedStatus,
        'Not Shared' AS SharedPercentOpen,
        CAST(NULL AS DATE) AS EffectiveDate,
        CAST(NULL AS DATE) AS ExpirationDate,
        CAST(NULL AS DECIMAL(18,2)) AS OrderedWgt,
        CAST(NULL AS DECIMAL(18,2)) AS ShippedWgt,
        COALESCE(UnshippedWgt, 0) AS PriorUnshippedWgt,
        COALESCE(UnshippedWgt, 0) AS UnshippedWgt,
        COALESCE(InTransitWgt, 0) AS PriorInTransitWgt,
        COALESCE(InTransitWgt, 0) AS InTransitWgt,
        CAST(NULL AS DECIMAL(18,2)) AS ReceivedWgt,
        CAST(NULL AS DECIMAL(18,2)) AS DueWgt,
        CAST(NULL AS DECIMAL(18,4)) AS OrderedAmount,
        CAST(NULL AS DECIMAL(18,4)) AS ReceivedAmount,
        UnshippedAmount AS PriorUnshippedAmount,
        UnshippedAmount,
        InTransitAmount AS PriorInTransitAmount,
        InTransitAmount,
        CAST(NULL AS DECIMAL(18,4)) AS MaterialCostGT,
        CAST(NULL AS DECIMAL(18,4)) AS DJJCost,
        CAST(NULL AS DECIMAL(18,4)) AS CommissionAmount,
        CAST(NULL AS DECIMAL(18,4)) AS NucorCost,
        0 AS DJJGeneratedFlag,
        {ETLID} AS ETLSSExecutionID,
        current_timestamp() AS DJJLastUpdateTime
    FROM {federated_starschema_catalog}.djj_brk.factOpenOrderSnapShot a
    LEFT OUTER JOIN (
        SELECT MAX(DateKey) AS DateKey, MAX(Day_of_Week) AS Max_Day_Of_Week, Fiscal_Week_of_Year
        FROM {federated_starschema_catalog}.djj.dimDate
        GROUP BY Fiscal_Week_of_Year
    ) e ON a.DateKey = e.DateKey
    INNER JOIN {federated_starschema_catalog}.djj_brk.dimConsumer c
        ON a.ConsumerKey = c.ConsumerKey
    INNER JOIN {federated_starschema_catalog}.Nue.dimNucorMillLocations d
        ON c.ConsumerID = d.DJJConsumerID
    INNER JOIN {federated_starschema_catalog}.djj_brk.dimGrade f
        ON a.GradeKey = f.GradeKey
    INNER JOIN {federated_starschema_catalog}.djj_brk.dimInventoryLocation g
        ON a.InventoryLocationKey = g.InventoryLocationKey
    INNER JOIN {federated_starschema_catalog}.djj_brk.dimCargo car
        ON a.CargoKey = car.CargoKey
    WHERE lower(trim(ContractNum)) LIKE lower(trim('%MDS%'))
    AND a.SourceSystemKey = 7
    AND e.DateKey IS NOT NULL
""")

In [ ]:
# Cell 9: Update ShippingScenarioID for Nucor Unallocated contracts
# UPDATE with FROM -> MERGE INTO
# StatusDesc NOT IN ('Cancelled','Finished') - static literal list kept as-is
spark.sql(f"""
    MERGE INTO {target_catalog}.temp.NucorMDSCargoAdjustments a
    USING (
        SELECT
            tmp.FiscalWeekKey,
            tmp.DateKey,
            tmp.BizType,
            tmp.ContractType,
            tmp.ContractNum,
            tmp.ContractLine,
            tmp.SourceSystemKey,
            tmp.NucorOrgID,
            tmp.ShippingScenarioID AS orig_ShippingScenarioID,
            capa.ShippingScenarioID AS new_ShippingScenarioID,
            capa.StatusDesc AS new_StatusDesc
        FROM {target_catalog}.temp.NucorMDSCargoAdjustments tmp
        LEFT OUTER JOIN (
            SELECT DISTINCT CargoID, ShippingScenarioID
            FROM {federated_starschema_catalog}.Nue.factCargoMillDetails
            WHERE NucorLocationKey = 23
            AND lower(trim(StatusDesc)) NOT IN (lower(trim('Cancelled')), lower(trim('Finished')))
        ) cmd ON tmp.CargoID = cmd.CargoID
        LEFT OUTER JOIN (
            SELECT DISTINCT FiscalWeekKey, CargoID, ShippingScenarioID, StatusDesc
            FROM {federated_starschema_catalog}.Nue.factCargoAllocationPowerApps
        ) capa
            ON tmp.FiscalWeekKey = capa.FiscalWeekKey
            AND tmp.CargoID = capa.CargoID
        WHERE tmp.NucorOrgID = -2
        AND cmd.CargoID IS NOT NULL
        AND capa.CargoID IS NOT NULL
    ) b
    ON a.FiscalWeekKey = b.FiscalWeekKey
    AND a.DateKey = b.DateKey
    AND lower(trim(a.BizType)) = lower(trim(b.BizType))
    AND lower(trim(a.ContractType)) = lower(trim(b.ContractType))
    AND lower(trim(a.ContractNum)) = lower(trim(b.ContractNum))
    AND a.ContractLine = b.ContractLine
    AND a.SourceSystemKey = b.SourceSystemKey
    AND a.NucorOrgID = b.NucorOrgID
    AND a.ShippingScenarioID = b.orig_ShippingScenarioID
    WHEN MATCHED THEN UPDATE SET
        a.ShippingScenarioID = b.new_ShippingScenarioID,
        a.StatusDesc = b.new_StatusDesc
""")

In [ ]:
# Cell 10: UPSERT - MERGE INTO target factOpenOrderSnapshot (UPDATE + INSERT)
# Original SQL UPDATE with FROM + INSERT with LEFT OUTER JOIN -> single MERGE INTO
# 9-column composite key: FiscalWeekKey, DateKey, BizType, ContractType, ContractNum, ContractLine, SourceSystemKey, NucorOrgID, ShippingScenarioID
spark.sql(f"""
    MERGE INTO {target_catalog}.Nue.factOpenOrderSnapshot target
    USING {target_catalog}.temp.NucorMDSCargoAdjustments source
    ON target.FiscalWeekKey = source.FiscalWeekKey
    AND target.DateKey = source.DateKey
    AND lower(trim(target.BizType)) = lower(trim(source.BizType))
    AND lower(trim(target.ContractType)) = lower(trim(source.ContractType))
    AND lower(trim(target.ContractNum)) = lower(trim(source.ContractNum))
    AND target.ContractLine = source.ContractLine
    AND target.SourceSystemKey = source.SourceSystemKey
    AND target.NucorOrgID = source.NucorOrgID
    AND target.ShippingScenarioID = source.ShippingScenarioID
    WHEN MATCHED THEN UPDATE SET
        target.InputDateKey = source.InputDateKey,
        target.OrderDateKey = source.OrderDateKey,
        target.BuyPlanMonthKey = source.BuyPlanMonthKey,
        target.GradeKey = source.GradeKey,
        target.SupplierID = source.SupplierID,
        target.ConsumerID = source.ConsumerID,
        target.GradeCode = source.GradeCode,
        target.OrderPropertiesKey = source.OrderPropertiesKey,
        target.InventoryLocationKey = source.InventoryLocationKey,
        target.VehicleTypeKey = source.VehicleTypeKey,
        target.CargoID = source.CargoID,
        target.StatusDesc = source.StatusDesc,
        target.SharedStatus = source.SharedStatus,
        target.SharedPercentOpen = source.SharedPercentOpen,
        target.EffectiveDate = source.EffectiveDate,
        target.ExpirationDate = source.ExpirationDate,
        target.OrderedWgt = source.OrderedWgt,
        target.ShippedWgt = source.ShippedWgt,
        target.PriorUnshippedWgt = source.PriorUnshippedWgt,
        target.UnshippedWgt = source.UnshippedWgt,
        target.PriorInTransitWgt = source.PriorInTransitWgt,
        target.InTransitWgt = source.InTransitWgt,
        target.ReceivedWgt = source.ReceivedWgt,
        target.DueWgt = source.DueWgt,
        target.OrderedAmount = source.OrderedAmount,
        target.ReceivedAmount = source.ReceivedAmount,
        target.PriorUnshippedAmount = source.PriorUnshippedAmount,
        target.UnshippedAmount = source.UnshippedAmount,
        target.PriorInTransitAmount = source.PriorInTransitAmount,
        target.InTransitAmount = source.InTransitAmount,
        target.MaterialCostGT = source.MaterialCostGT,
        target.DJJCost = source.DJJCost,
        target.CommissionAmount = source.CommissionAmount,
        target.NucorCost = source.NucorCost,
        target.DJJGeneratedFlag = source.DJJGeneratedFlag,
        target.ETLSSExecutionID = source.ETLSSExecutionID,
        target.DJJLastUpdateTime = source.DJJLastUpdateTime
    WHEN NOT MATCHED THEN INSERT (
        FiscalWeekKey, DateKey, BizType, ContractType, ContractNum, ContractLine,
        SourceSystemKey, NucorOrgID, ShippingScenarioID, InputDateKey, OrderDateKey,
        BuyPlanMonthKey, GradeKey, SupplierID, ConsumerID, GradeCode, OrderPropertiesKey,
        InventoryLocationKey, VehicleTypeKey, CargoID, StatusDesc, SharedStatus, SharedPercentOpen,
        EffectiveDate, ExpirationDate, OrderedWgt, ShippedWgt, PriorUnshippedWgt,
        UnshippedWgt, PriorInTransitWgt, InTransitWgt, ReceivedWgt, DueWgt,
        OrderedAmount, ReceivedAmount, PriorUnshippedAmount, UnshippedAmount,
        PriorInTransitAmount, InTransitAmount, MaterialCostGT, DJJCost,
        CommissionAmount, NucorCost, DJJGeneratedFlag, ETLSSExecutionID, DJJLastUpdateTime
    ) VALUES (
        source.FiscalWeekKey, source.DateKey, source.BizType, source.ContractType,
        source.ContractNum, source.ContractLine, source.SourceSystemKey, source.NucorOrgID,
        source.ShippingScenarioID, source.InputDateKey, source.OrderDateKey,
        source.BuyPlanMonthKey, source.GradeKey, source.SupplierID, source.ConsumerID,
        source.GradeCode, source.OrderPropertiesKey, source.InventoryLocationKey,
        source.VehicleTypeKey, source.CargoID, source.StatusDesc, source.SharedStatus,
        source.SharedPercentOpen, source.EffectiveDate, source.ExpirationDate,
        source.OrderedWgt, source.ShippedWgt, source.PriorUnshippedWgt, source.UnshippedWgt,
        source.PriorInTransitWgt, source.InTransitWgt, source.ReceivedWgt, source.DueWgt,
        source.OrderedAmount, source.ReceivedAmount, source.PriorUnshippedAmount,
        source.UnshippedAmount, source.PriorInTransitAmount, source.InTransitAmount,
        source.MaterialCostGT, source.DJJCost, source.CommissionAmount, source.NucorCost,
        source.DJJGeneratedFlag, source.ETLSSExecutionID, source.DJJLastUpdateTime
    )
""")

In [ ]:
# Cell 11: DELETE - Remove MDS contracts not present in source Brk.factOpenOrderSnapShot
# DELETE with LEFT JOIN WHERE IS NULL -> MERGE INTO with DELETE
spark.sql(f"""
    MERGE INTO {target_catalog}.Nue.factOpenOrderSnapshot target
    USING (
        SELECT a.FiscalWeekKey, a.DateKey, a.BizType, a.ContractType, a.ContractNum,
               a.ContractLine, a.SourceSystemKey, a.NucorOrgID, a.ShippingScenarioID
        FROM {target_catalog}.Nue.factOpenOrderSnapshot a
        LEFT OUTER JOIN {federated_starschema_catalog}.djj_brk.factOpenOrderSnapShot b
            ON a.DateKey = b.DateKey
            AND lower(trim(a.ContractType)) = lower(trim(CASE WHEN lower(trim(b.ContractType)) = lower(trim('P')) THEN 'Purchase' ELSE 'Sale' END))
            AND a.ShippingScenarioID = b.ShippingScenarioID
            AND lower(trim(a.ContractNum)) = lower(trim(b.ContractNum))
            AND a.ContractLine = b.ContractLine
            AND a.SourceSystemKey = b.SourceSystemKey
            AND lower(trim(a.BizType)) = lower(trim(b.Company))
        WHERE a.SourceSystemKey = 7
        AND b.DateKey IS NULL
    ) source
    ON target.FiscalWeekKey = source.FiscalWeekKey
    AND target.DateKey = source.DateKey
    AND lower(trim(target.BizType)) = lower(trim(source.BizType))
    AND lower(trim(target.ContractType)) = lower(trim(source.ContractType))
    AND lower(trim(target.ContractNum)) = lower(trim(source.ContractNum))
    AND target.ContractLine = source.ContractLine
    AND target.SourceSystemKey = source.SourceSystemKey
    AND target.NucorOrgID = source.NucorOrgID
    AND target.ShippingScenarioID = source.ShippingScenarioID
    WHEN MATCHED THEN DELETE
""")

In [ ]:
# Cell 12: DELETE - Remove data not for last day of the fiscal week
# DELETE with LEFT JOIN WHERE IS NULL -> MERGE INTO with DELETE
spark.sql(f"""
    MERGE INTO {target_catalog}.Nue.factOpenOrderSnapshot target
    USING (
        SELECT a.FiscalWeekKey, a.DateKey, a.BizType, a.ContractType, a.ContractNum,
               a.ContractLine, a.SourceSystemKey, a.NucorOrgID, a.ShippingScenarioID
        FROM {target_catalog}.Nue.factOpenOrderSnapshot a
        LEFT OUTER JOIN (
            SELECT MAX(DateKey) AS DateKey, MAX(Day_of_Week) AS Max_Day_Of_Week, Fiscal_Week_of_Year
            FROM {federated_starschema_catalog}.djj.dimDate
            GROUP BY Fiscal_Week_of_Year
        ) e ON a.DateKey = e.DateKey
        WHERE a.SourceSystemKey = 7
        AND e.DateKey IS NULL
    ) source
    ON target.FiscalWeekKey = source.FiscalWeekKey
    AND target.DateKey = source.DateKey
    AND lower(trim(target.BizType)) = lower(trim(source.BizType))
    AND lower(trim(target.ContractType)) = lower(trim(source.ContractType))
    AND lower(trim(target.ContractNum)) = lower(trim(source.ContractNum))
    AND target.ContractLine = source.ContractLine
    AND target.SourceSystemKey = source.SourceSystemKey
    AND target.NucorOrgID = source.NucorOrgID
    AND target.ShippingScenarioID = source.ShippingScenarioID
    WHEN MATCHED THEN DELETE
""")

In [ ]:
# Cell 13: DELETE - Temporary cleanup of duplicate MDS records for Nucor Unallocated
# DELETE with INNER JOIN -> MERGE INTO with DELETE
spark.sql(f"""
    MERGE INTO {target_catalog}.Nue.factOpenOrderSnapshot target
    USING (
        SELECT ooss.DateKey, lower(trim(ooss.ContractNum)) AS ContractNum, ooss.ContractLine, ooss.ShippingScenarioID
        FROM {target_catalog}.Nue.factOpenOrderSnapshot ooss
        LEFT OUTER JOIN (
            SELECT DateKey, ContractNum, ContractLine, COUNT(*) AS Total
            FROM {target_catalog}.Nue.factOpenOrderSnapshot
            WHERE lower(trim(ContractNum)) LIKE lower(trim('%MDS%'))
            AND NucorOrgID = -2
            GROUP BY DateKey, ContractNum, ContractLine
            HAVING COUNT(*) > 1
        ) x
            ON ooss.DateKey = x.DateKey
            AND lower(trim(ooss.ContractNum)) = lower(trim(x.ContractNum))
            AND ooss.ContractLine = x.ContractLine
        WHERE x.DateKey IS NOT NULL
        AND ooss.ShippingScenarioID = 0
    ) source
    ON target.DateKey = source.DateKey
    AND lower(trim(target.ContractNum)) = lower(trim(source.ContractNum))
    AND target.ContractLine = source.ContractLine
    AND target.ShippingScenarioID = 0
    AND target.SourceSystemKey = 7
    WHEN MATCHED THEN DELETE
""")

In [ ]:
# Cell 14: Clean up Temp Tables
spark.sql(f"DROP TABLE IF EXISTS {target_catalog}.temp.NucorMDSCargoAdjustments")

In [ ]:
# Cell 15: Update records in ETLExecutionLog table
exec_logger.log_execution(
    ETLID=ETLID,
    ETLRunTime=ETLRunTime,
    Success='1',
    RowsInserted=RowsInserted,
    RowsUpdated=RowsUpdated,
    ErrorRowCount=ErrorRowCount,
    TableInitialRowCount=TableInitialRowCount,
    TableFinalRowCount=TableFinalRowCount,
    ETLStatus=ETLStatus,
    ExtractRowCount=ExtractRowCount,
    SSISGUID=SSIDGUID
)

In [ ]:
# Cell 16: Update ETLMaster with final metadata
spark.sql(f"""UPDATE djj_enriched_non_prod.metadata.ETLMaster
    SET IncrementalExtractTime = lower(trim('{IncrementalExtractTime}')),
        ETLTemplate = lower(trim('{ETLTemplate}')),
        ETLType = lower(trim('{ETLType}'))
    WHERE ETLID = lower(trim('{ETLID}'))""")